In [122]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/datasets', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [123]:
import pandas as pd
import glob
import numpy as np
import joblib
import pickle
import ast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from sklearn.utils import resample
import tensorflow as tf
import csv
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, Dropout, TextVectorization, GlobalMaxPooling1D, Conv1D, SpatialDropout1D, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import nltk
from nltk.corpus import stopwords
import re

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [124]:
all_files = glob.glob("/content/drive/MyDrive/datasets/*.csv")

df = pd.concat([
    pd.read_csv(f, engine='python', quoting=csv.QUOTE_MINIMAL, on_bad_lines='skip')
    for f in all_files
], ignore_index=True)

print(f"Total rows: {len(df)}")
print(f"Dari {len(all_files)} file: {[os.path.basename(f) for f in all_files]}")
df.info()

Total rows: 8236
Dari 1 file: ['df_merge.csv']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8236 entries, 0 to 8235
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                8236 non-null   float64
 1   title             8236 non-null   object 
 2   search_role       8236 non-null   object 
 3   job_level         8236 non-null   object 
 4   company           8236 non-null   object 
 5   location          5902 non-null   object 
 6   salary_avg        1667 non-null   float64
 7   extracted_skills  8236 non-null   object 
 8   skills_count      8236 non-null   int64  
 9   job_url           8236 non-null   object 
 10  scraped_at        8236 non-null   object 
 11  job_description   8231 non-null   object 
dtypes: float64(2), int64(1), object(9)
memory usage: 772.3+ KB


In [125]:
df = df.drop_duplicates()
df = df.drop(columns=['salary', 'search_role_raw', 'job_level', 'location_raw',
                       'salary_display', 'salary_min', 'salary_max', 'salary_avg', 'company', 'location', 'job_url', 'search_location', 'scraped_at'],
             errors='ignore')
print(f"Total duplicated rows: {df.duplicated().sum()}")
df.dropna(inplace=True)
print(df.isnull().sum())

Total duplicated rows: 155
id                  0
title               0
search_role         0
extracted_skills    0
skills_count        0
job_description     0
dtype: int64


In [126]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8231 entries, 0 to 8235
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                8231 non-null   float64
 1   title             8231 non-null   object 
 2   search_role       8231 non-null   object 
 3   extracted_skills  8231 non-null   object 
 4   skills_count      8231 non-null   int64  
 5   job_description   8231 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 450.1+ KB


In [127]:
def parse_skills(val):
    try:
        result = ast.literal_eval(val)
        return ' '.join(result) if isinstance(result, list) and len(result) > 0 else ''
    except:
        return ''

df['extracted_skills_clean'] = df['extracted_skills'].apply(parse_skills)

df['text_input'] = df['title'] + ' ' + df['job_description'] + ' ' + df['extracted_skills_clean']
df['text_input'] = df['text_input'].fillna('').astype(str)

df = df[df['text_input'].str.strip() != ''].reset_index(drop=True)

In [128]:
min_samples = 200
role_counts = df['search_role'].value_counts()
valid_roles = role_counts[role_counts >= min_samples].index
df = df[df['search_role'].isin(valid_roles)].reset_index(drop=True)
print(df['search_role'].value_counts())

search_role
Backend Developer      1374
Fullstack Developer    1279
DevOps Engineer        1254
Data Analyst           1088
Frontend Developer     1024
Data Engineer          1005
Software Engineer       492
ML Engineer             399
Data Scientist          316
Name: count, dtype: int64


In [129]:
df['search_role'] = df['search_role'].replace({'Full stack Developer': 'Fullstack Developer'})

roles_to_drop = ['Software Engineer', 'Web Developer', 'IT Support', 'IT',
                 'Developer', 'Software', 'System Analyst', 'Progammer', 'Software Developer']
df = df[~df['search_role'].isin(roles_to_drop)]

min_samples = 200
role_counts = df['search_role'].value_counts()
valid_roles = role_counts[role_counts >= min_samples].index
df = df[df['search_role'].isin(valid_roles)].reset_index(drop=True)

print(df['search_role'].value_counts())

search_role
Backend Developer      1374
Fullstack Developer    1279
DevOps Engineer        1254
Data Analyst           1088
Frontend Developer     1024
Data Engineer          1005
ML Engineer             399
Data Scientist          316
Name: count, dtype: int64


In [130]:
encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['search_role'])
num_classes = len(encoder.classes_)

df['text_input'] = df['text_input'].fillna('').astype(str)

print(f"Classes: {encoder.classes_}")
print(f"Num classes: {num_classes}")

Classes: ['Backend Developer' 'Data Analyst' 'Data Engineer' 'Data Scientist'
 'DevOps Engineer' 'Frontend Developer' 'Fullstack Developer'
 'ML Engineer']
Num classes: 8


In [131]:
stop_words_en = set(stopwords.words('english'))
stop_words_id = set(stopwords.words('indonesian'))

all_stop_words = stop_words_en.union(stop_words_id)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)

    words = text.split()
    clean_words = [w for w in words if w not in all_stop_words]

    return ' '.join(clean_words)

df['text_input'] = df['text_input'].apply(clean_text)

In [132]:
print("=== Label alignment check ===")
for i in range(5):
    print(f"Role: {df['search_role'].iloc[i]}")
    print(f"Label: {df['label'].iloc[i]}")
    print(f"Title: {df['title'].iloc[i]}")
    print()

=== Label alignment check ===
Role: ML Engineer
Label: 7
Title: AI & Automation Engineer

Role: ML Engineer
Label: 7
Title: AI Officer (Part-Time/Project-Based)

Role: ML Engineer
Label: 7
Title: Engineering Staff

Role: ML Engineer
Label: 7
Title: Data Engineer

Role: ML Engineer
Label: 7
Title: Programmable Logic Controller (PLC) Automation Engineer



In [133]:
X_train, X_val, y_train, y_val = train_test_split(
    df['text_input'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label']
)

train_data = pd.DataFrame({'text': X_train, 'label': y_train})

target_samples = 800
dfs_resampled = []

for role_label in train_data['label'].unique():
    df_role = train_data[train_data['label'] == role_label]
    if len(df_role) < target_samples:
        df_role = resample(df_role, replace=True, n_samples=target_samples, random_state=42)
    dfs_resampled.append(df_role)

train_data_balanced = pd.concat(dfs_resampled).sample(frac=1, random_state=42).reset_index(drop=True)

X_train_bal = train_data_balanced['text']
y_train_bal = train_data_balanced['label']

print("Distribusi setelah resampling:")
print(pd.Series(y_train_bal).value_counts().sort_index())
print(f"\nTotal training samples: {len(X_train_bal)}")
print(f"Total val samples: {len(X_val)}")

vectorizer = TextVectorization(
    max_tokens=15000,
    output_mode='int',
    output_sequence_length=300,
    ngrams=2
)
vectorizer.adapt(X_train_bal.to_numpy())
print(f"Vocab size: {len(vectorizer.get_vocabulary())}")

Distribusi setelah resampling:
label
0    1099
1     871
2     804
3     800
4    1003
5     819
6    1023
7     800
Name: count, dtype: int64

Total training samples: 7219
Total val samples: 1548
Vocab size: 15000


In [134]:
print("=== Sample text inputs ===")
for role in df['search_role'].unique():
    sample = df[df['search_role'] == role]['text_input'].iloc[0]
    print(f"\n[{role}]")
    print(sample[:200])

print("\n=== Vectorizer vocab sample ===")
print(vectorizer.get_vocabulary()[:50])
print(f"\nVocab size: {len(vectorizer.get_vocabulary())}")

=== Sample text inputs ===

[ML Engineer]
ai automation engineer part system automation team support development enhancement ai driven automation solutions improve service delivery operations includes leveraging generative ai genai low code p

[Backend Developer]
coding teacher elementary middle school part time fbs career coding teacher elementary middle school part time opening date june closing date tba employment code els ms required qualifications minimum

[Data Analyst]
commercial superintendent key responsibilities support implementation commercial strategies across coal mining epc projects assist contract administration including preparation review monitoring comme

[Data Engineer]
admin support staff qualification education experience bachelor degree business administration management information systems industrial engineering statistics related fields b minimum years experienc

[Data Scientist]
data analyst admin staff work description olah data materi rakor bulanan rekap per

In [135]:
class CustomEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, patience=5):
        super().__init__()
        self.patience = patience
        self.best_weights = None
        self.best_loss = np.inf
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get('val_loss')
        if current_loss < self.best_loss:
            self.best_loss = current_loss
            self.wait = 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\n[INFO] val_loss tidak membaik selama {self.patience} epoch. Stop training!")
                self.model.stop_training = True
                print("[INFO] Mengembalikan bobot model ke epoch terbaik.")
                self.model.set_weights(self.best_weights)

callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    CustomEarlyStopping(patience=5)
]

In [136]:
vocab_size = len(vectorizer.get_vocabulary())

inputs = Input(shape=(1,), dtype=tf.string, name='text_input')
x = vectorizer(inputs)
x = Embedding(input_dim=vocab_size, output_dim=64)(x)
x = SpatialDropout1D(0.3)(x)
x = Conv1D(128, 5, activation='relu')(x)
x = GlobalMaxPooling1D()(x)
x = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(x)
x = Dropout(0.5)(x)

outputs = Dense(num_classes, activation='softmax', name='role_output')(x)

model = Model(inputs=inputs, outputs=outputs, name="Job_Role_Classifier")
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

print("Mulai proses training...")
history = model.fit(
    X_train_bal.to_numpy(), y_train_bal,
    validation_data=(X_val.to_numpy(), y_val),
    epochs=40,
    batch_size=64,
    callbacks=callbacks
)

Model: "Job_Role_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_input (InputLayer)         │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_7            │ (None, 300)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 300, 64)        │       960,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_2             │ (None, 300, 64)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 296, 128)       │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_2          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ role_output (Dense)             │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,009,864 (3.85 MB)

 Trainable params: 1,009,864 (3.85 MB)

 Non-trainable params: 0 (0.00 B)

Mulai proses training...
Epoch 1/40
113/113 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.1763 - loss: 2.0990 - val_accuracy: 0.2519 - val_loss: 1.9153 - learning_rate: 0.0010
Epoch 2/40
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.3599 - loss: 1.6887 - val_accuracy: 0.4935 - val_loss: 1.4687 - learning_rate: 0.0010
Epoch 3/40
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5063 - loss: 1.3464 - val_accuracy: 0.5200 - val_loss: 1.2861 - learning_rate: 0.0010
Epoch 4/40
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5902 - loss: 1.1202 - val_accuracy: 0.5620 - val_loss: 1.1912 - learning_rate: 0.0010
Epoch 5/40
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.6531 - loss: 0.9339 - val_accuracy: 0.5736 - val_loss: 1.1407 - learning_rate: 0.0010
Epoch 6/40
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.7000 - loss: 0.8126 - val_accuracy: 0.5788 - val_loss: 1.1173 - learning_rate: 0.0010
Epoch 7/40
113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - ac

In [137]:
y_pred = np.argmax(model.predict(X_val.to_numpy()), axis=1)
print(classification_report(y_val, y_pred, target_names=encoder.classes_))

49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
                     precision    recall  f1-score   support

  Backend Developer       0.47      0.88      0.61       275
       Data Analyst       0.61      0.71      0.66       217
      Data Engineer       0.72      0.84      0.77       201
     Data Scientist       0.59      0.41      0.49        63
    DevOps Engineer       0.65      0.51      0.57       251
 Frontend Developer       0.00      0.00      0.00       205
Fullstack Developer       0.61      0.52      0.56       256
        ML Engineer       0.54      0.54      0.54        80

           accuracy                           0.58      1548
          macro avg       0.52      0.55      0.52      1548
       weighted avg       0.52      0.58      0.53      1548



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [138]:
import joblib
from google.colab import files

model.save('job_role_model.keras')
print("[INFO] Model berhasil disimpan")

joblib.dump(encoder, 'label_encoder.pkl')
print("[INFO] Label encoder berhasil disimpan")

# Test inference dengan input skill aja (simulasi kondisi production)
loaded_model = tf.keras.models.load_model('job_role_model.keras')
loaded_encoder = joblib.load('label_encoder.pkl')

def rekomendasi_role(skills_text):
    input_tensor = tf.constant([skills_text], dtype=tf.string)
    pred_probs = loaded_model.predict(input_tensor, verbose=0)
    pred_class_idx = np.argmax(pred_probs)
    predicted_role = loaded_encoder.inverse_transform([pred_class_idx])[0]
    confidence = float(np.max(pred_probs))
    return predicted_role, confidence

# Test dengan skill aja, TANPA title/job desc
skill_input = "experience building REST APIs Laravel FastAPI managing databases PostgreSQL"
role, conf = rekomendasi_role(skill_input)
print(f"\n--- Hasil Inference ---")
print(f"Skill Input : {skill_input}")
print(f"Cocok untuk : {role} ({conf:.1%})")

[INFO] Model berhasil disimpan
[INFO] Label encoder berhasil disimpan

--- Hasil Inference ---
Skill Input : experience building REST APIs Laravel FastAPI managing databases PostgreSQL
Cocok untuk : Backend Developer (39.8%)
